In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

Dataset Loading, Device Setup, and Basic Tokenizer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset   # برای لود کردن دیتاست IMDB
from torch.utils.data import DataLoader   # برای ساخت دیتالودر

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


train_data = dataset["train"]
test_data = dataset["test"]
def tokenize(text):
    return text.lower().split()

Vocabulary Construction and Token Frequency Counting

In [ ]:
from collections import Counter
counter = Counter()

for i in train_data:
    counter.update(tokenize(i["text"]))

# فقط 20 هزار کلمهٔ پرتکرار را نگه می‌داریم
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(20000))}

# دو توکن خاص
vocab["<unk>"] = 1   # کلمه ناشناخته
vocab["<pad>"] = 0   # پدینگ

Encoding Function for Token-to-ID Conversion

In [ ]:
def encode(text):
    tokens = tokenize(text)   # متن را به کلمات تبدیل می‌کنیم
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens]    # هر کلمه را تبدیل به عدد می‌کنیم
    return torch.tensor(ids)

Batch Collation and DataLoader Setup

In [ ]:
def collate_fn(batch):
    texts = [encode(i["text"]) for i in batch]    # تبدیل هر متن به Tensor
    labels = torch.tensor([i["label"] for i in batch])    # تبدیل label به Tensor

    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)    # پدینگ برای یکسان کردن طول جمله‌ها
    return texts,labels

train_loader = DataLoader(train_data, batch_size=128, shuffle=True , collate_fn=collate_fn)
test_data = DataLoader(test_data, batch_size=128 , shuffle=False ,collate_fn=collate_fn)


RNN Model Definition (PyTorch)

In [ ]:
class RNN(nn.Module):
    def __init__(self,vocab_size,embed_dim,hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)   
        self.rnn = nn.RNN(embed_dim,hidden_dim,batch_first=True)      
        self.fc = nn.Linear(hidden_dim,2)     
    def forward(self,x):
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        _,hidden = self.rnn(embedded)  # hidden شکل (1, batch, hidden_dim)
        return self.fc(hidden.squeeze(0))

Model Initialization and Training Setup

In [ ]:
model = RNN(len(vocab),64,128)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

Training Loop for RNN Model

In [ ]:
for epoch in range(8):
    total_loss = 0
    for texts,labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    avg_loss = total_loss / len(train_loader)
    print(avg_loss)

Prediction Function for Inference

In [ ]:
def predict(text):
    model.eval()
    with torch.no_grad():
        ids = encode(text).unsqueeze(0).to(device)
        output = model(ids)
        pred = torch.argmax(output).item()
        return "+" if pred ==1 else "-"

Inference Test Prints

In [ ]:
print(predict("This movie was amazing"))
print(predict("Worst movie ever"))